# Evaluación comparable de los modelos predictivos

Este notebook evalúa Maxent, GLM, Random Forest y XGBoost con la **misma muestra, las mismas particiones y las mismas métricas** (Cuadro 3 de la tesis).

**Unidad de observación**: sitios candidatos con resultado, construidos en `deduplicacion_sitios.ipynb` (`sitios.csv`): sin cementerios, dentro de su municipio y con $D_i$ observado. Las covariables se miden en el centroide del sitio. Los modelos también producen puntajes fuera de municipio para los sitios candidatos sin resultado, que se usan en la simulación territorial de `evaluacion_politicas.ipynb`.

Decisiones de evaluación:
- **Fuera de municipio**: validación dejando un municipio por fuera. Las métricas se promedian sobre los municipios que tienen presencias *y* ausencias (en los que solo tienen presencias, P@k = 1 por construcción y el AUC no está definido). También se reportan las métricas agrupando todas las predicciones fuera de municipio.
- **Reserva de municipios**: Samaná, San Onofre y Peñol.
- **División 80/20**: estratificada, `random_state=42`.
- **Métricas**: AUC, P@k, Recall@k, Lift@k (P@k / prevalencia de la muestra de evaluación) y Brier, con k = 30% de las observaciones de prueba. P@k y Recall@k tratan los empates en el umbral como un desempate aleatorio (valor esperado), de modo que no dependen del orden de las filas.
- **Hiperparámetros**: los de los modelos finales de cada notebook.
- **Dos versiones de Maxent**: (i) *prospecciones negativas*, que contrasta las presencias con las prospecciones sin hallazgo (los mismos datos que los clasificadores); y (ii) *fondo del territorio*, que contrasta las presencias con puntos muestreados al azar dentro de los municipios **de entrenamiento** de cada partición. Ambas se evalúan sobre las mismas diligencias con resultado. Como la salida de Maxent con fondo no es una probabilidad de hallazgo en la muestra intervenida, su puntaje de Brier no se reporta.


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import elapid
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss

PATH_PROCESSED  = "../data/Processed/"
PATH_MUNICIPIOS = "../data/Municipios_Diciembre_2025_shp/Munpio.shp"
PATH_NDVI       = "../data/landsat/landsat_ndvi/"
PATH_DEM        = "../data/dem/dem_srtm/"
CRS             = "EPSG:4326"

FRAC_K = 0.3
MUNICIPIOS_HOLDOUT = ['17662', '70713', '05543']  # Samaná, San Onofre, Peñol
N_FONDO_POR_MUNICIPIO = 300
SEMILLA = 42


In [ ]:
# ============================================================
# CARGAR SITIOS Y COVARIABLES
# ============================================================
municipios = gpd.read_file(PATH_MUNICIPIOS)
municipios['MpCodigo'] = municipios['MpCodigo'].astype(str).str.zfill(5)
municipios = municipios.to_crs(CRS)

nombres_covariables = [
    'ndvi', 'altitud', 'pendiente',
    'dist_hidrografia', 'dist_vias',
    'dist_centros_poblados', 'dist_grupos_armados'
]

def extraer_valor_raster(raster_path, lon, lat):
    try:
        with rasterio.open(raster_path) as src:
            val = list(src.sample([(lon, lat)]))[0][0]
            if val == src.nodata:
                return np.nan
            return float(val)
    except:
        return np.nan

def calcular_pendiente_punto(dem_path, lon, lat, delta=0.001):
    try:
        with rasterio.open(dem_path) as src:
            z   = list(src.sample([(lon, lat)]))[0][0]
            z_e = list(src.sample([(lon+delta, lat)]))[0][0]
            z_n = list(src.sample([(lon, lat+delta)]))[0][0]
            dz_dx = (z_e - z) / (delta * 111320)
            dz_dy = (z_n - z) / (delta * 111320)
            return float(np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))))
    except:
        return np.nan

sitios = pd.read_csv(Path(PATH_PROCESSED) / "sitios.csv", dtype={'municipio': str})
candidatos = sitios[(sitios['tipo_sitio'] == 'otro') & (~sitios['fuera_municipio'])].reset_index(drop=True)

X_list = []
for codigo, g in candidatos.groupby('municipio', sort=False):
    ndvi_path = Path(PATH_NDVI) / f"ndvi_{codigo}.tif"
    dem_path  = Path(PATH_DEM)  / f"dem_{codigo}.tif"
    stack = np.load(Path(PATH_PROCESSED) / f"covariables_{codigo}.npy", mmap_mode='r')
    bounds = municipios[municipios['MpCodigo'] == codigo].total_bounds
    h, w = stack.shape[1], stack.shape[2]
    for i, sitio in g.iterrows():
        lon, lat = sitio['lon'], sitio['lat']
        col = np.clip(int((lon - bounds[0]) / (bounds[2] - bounds[0]) * w), 0, w-1)
        row = np.clip(int((bounds[3] - lat) / (bounds[3] - bounds[1]) * h), 0, h-1)
        covs = np.array([extraer_valor_raster(ndvi_path, lon, lat),
                         extraer_valor_raster(dem_path, lon, lat),
                         calcular_pendiente_punto(dem_path, lon, lat),
                         *stack[3:7, row, col]], dtype=float)
        X_list.append((i, np.where(np.isnan(covs), 0.0, covs)))

X_list.sort(key=lambda t: t[0])
X_cand = np.array([c for _, c in X_list])

# Muestra de estimación: candidatos con resultado observado
observado = candidatos['D'].notna().values
muestra = candidatos[observado].reset_index(drop=True)
X      = X_cand[observado]
y      = muestra['D'].values.astype(float)
grupos = muestra['municipio'].values
coords = muestra[['lon', 'lat']].values

print(f"Sitios candidatos: {len(candidatos)}  |  sin resultado: {(~observado).sum()}")
print(f"Sitios con resultado: {len(X)}  |  con hallazgo: {int(y.sum())}  |  sin hallazgo: {int((y==0).sum())}")
print(f"Municipios: {len(np.unique(grupos))}  |  prevalencia: {y.mean():.3f}")
print(f"Municipios con hallazgos y no hallazgos: "
      f"{sum(y[grupos == m].min() < y[grupos == m].max() for m in np.unique(grupos))}")


In [ ]:
# ============================================================
# PUNTOS DE FONDO DEL TERRITORIO (para Maxent con fondo)
# ============================================================
# Se muestrean N_FONDO_POR_MUNICIPIO puntos al azar dentro de cada municipio con
# diligencias, con el mismo número por municipio para que el fondo contraste cada
# presencia con el territorio disponible en su municipio y no con el de los municipios
# más extensos. Solo se aceptan puntos con NDVI válido: 5 de las 657 diligencias no
# tienen NDVI, pero en la grilla falta en buena parte del territorio (nubes), e
# imputarlo con 0 en el fondo le enseñaría a Maxent que NDVI = 0 indica ausencia.
# Las covariables se extraen igual que para las diligencias.
from shapely.geometry import Point
from shapely.prepared import prep

PATH_FONDO = Path(PATH_PROCESSED) / "fondo_maxent.csv"

def muestrear_fondo(codigo, n, rng):
    ndvi_path = Path(PATH_NDVI) / f"ndvi_{codigo}.tif"
    dem_path  = Path(PATH_DEM)  / f"dem_{codigo}.tif"
    mun = municipios[municipios['MpCodigo'] == codigo]
    geom = prep(mun.geometry.values[0])
    bounds = mun.total_bounds
    stack = np.load(Path(PATH_PROCESSED) / f"covariables_{codigo}.npy", mmap_mode='r')
    h, w = stack.shape[1], stack.shape[2]
    filas = []
    with rasterio.open(ndvi_path) as src_ndvi, rasterio.open(dem_path) as src_dem:
        while len(filas) < n:
            lon = rng.uniform(bounds[0], bounds[2], 4 * n)
            lat = rng.uniform(bounds[1], bounds[3], 4 * n)
            dentro = np.array([geom.contains(Point(a, b)) for a, b in zip(lon, lat)])
            lon, lat = lon[dentro], lat[dentro]
            xy = list(zip(lon, lat))
            ndvi = np.array([v[0] for v in src_ndvi.sample(xy)], dtype=float)
            z    = np.array([v[0] for v in src_dem.sample(xy)], dtype=float)
            z_e  = np.array([v[0] for v in src_dem.sample([(a + 0.001, b) for a, b in xy])], dtype=float)
            z_n  = np.array([v[0] for v in src_dem.sample([(a, b + 0.001) for a, b in xy])], dtype=float)
            if src_ndvi.nodata is not None:
                ndvi[ndvi == src_ndvi.nodata] = np.nan
            if src_dem.nodata is not None:
                z[z == src_dem.nodata] = np.nan
            dz_dx = (z_e - z) / (0.001 * 111320)
            dz_dy = (z_n - z) / (0.001 * 111320)
            pendiente = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
            col = np.clip(((lon - bounds[0]) / (bounds[2] - bounds[0]) * w).astype(int), 0, w-1)
            row = np.clip(((bounds[3] - lat) / (bounds[3] - bounds[1]) * h).astype(int), 0, h-1)
            distancias = np.stack([stack[c][row, col] for c in range(3, 7)], axis=1)
            covs = np.column_stack([ndvi, z, pendiente, distancias])
            validos = ~np.isnan(ndvi)
            covs = np.where(np.isnan(covs), 0.0, covs)[validos]
            for (a, b), fila in zip(np.array(xy)[validos], covs):
                filas.append({'municipio': codigo, 'lon': a, 'lat': b,
                              **dict(zip(nombres_covariables, fila))})
    return filas[:n]

# Se cachea en disco; si aparecen municipios nuevos, se muestrean solo esos
df_fondo = (pd.read_csv(PATH_FONDO, dtype={'municipio': str}) if PATH_FONDO.exists()
            else pd.DataFrame(columns=['municipio']))
faltantes = sorted(set(grupos) - set(df_fondo['municipio']))
if faltantes:
    rng = np.random.default_rng(SEMILLA)
    nuevos = pd.DataFrame([f for codigo in faltantes
                           for f in muestrear_fondo(codigo, N_FONDO_POR_MUNICIPIO, rng)])
    df_fondo = pd.concat([df_fondo, nuevos], ignore_index=True)
    df_fondo.to_csv(PATH_FONDO, index=False)
df_fondo = df_fondo[df_fondo['municipio'].isin(np.unique(grupos))]

X_fondo = df_fondo[nombres_covariables].values
g_fondo = df_fondo['municipio'].values
print(f"Puntos de fondo: {len(df_fondo)} en {df_fondo['municipio'].nunique()} municipios")
print(f"Sitios con NDVI faltante (imputado en 0): {int((X[:, 0] == 0).sum())}")


In [ ]:
# ============================================================
# MÉTRICAS
# ============================================================
def positivos_esperados_top_k(y_true, y_score, k):
    """Positivos esperados en el top-k con desempate aleatorio en el umbral."""
    orden = np.sort(y_score)[::-1]
    umbral = orden[k - 1]
    arriba = y_score > umbral
    empate = y_score == umbral
    cupos = k - arriba.sum()
    return y_true[arriba].sum() + cupos * y_true[empate].mean()

def metricas(y_true, y_score, frac_k=FRAC_K):
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    k = max(1, int(len(y_true) * frac_k))
    prevalencia = y_true.mean()
    tp_k = positivos_esperados_top_k(y_true, y_score, k)
    prec_k = tp_k / k
    return {
        'auc': roc_auc_score(y_true, y_score) if 0 < prevalencia < 1 else np.nan,
        'precision_at_k': prec_k,
        'recall_at_k': tp_k / y_true.sum() if y_true.sum() > 0 else np.nan,
        'lift_at_k': prec_k / prevalencia if prevalencia > 0 else np.nan,
        'brier': brier_score_loss(y_true, y_score),
        'prevalencia': prevalencia,
        'n': len(y_true),
    }


In [ ]:
# ============================================================
# MODELOS (hiperparámetros de los modelos finales de cada notebook)
# ============================================================
class Modelo:
    """Interfaz común: fit(X, y, grupos) y score(X).

    Con usa_fondo=True, el modelo se entrena con las presencias de entrenamiento
    frente a los puntos de fondo de los municipios de entrenamiento, y no usa las
    prospecciones sin hallazgo."""
    def __init__(self, nombre, construir, score, usa_fondo=False):
        self.nombre, self.construir, self._score = nombre, construir, score
        self.usa_fondo = usa_fondo
    def fit(self, X, y, grupos_train):
        if self.usa_fondo:
            fondo = np.isin(g_fondo, np.unique(grupos_train))
            X = np.vstack([X[y == 1], X_fondo[fondo]])
            y = np.r_[np.ones((y == 1).sum()), np.zeros(fondo.sum())]
        self.m = self.construir(y)
        self.m.fit(X, y)
        return self
    def score(self, X):
        return self._score(self.m, X)

proba = lambda m, X: m.predict_proba(X)[:, 1]

maxent = lambda y: elapid.MaxentModel(feature_types=['linear'], beta_multiplier=1.5,
                                      transform='cloglog', scorer='roc_auc')
score_maxent = lambda m, X: np.asarray(m.predict(X), dtype=float)

MODELOS = [
    Modelo('Maxent (fondo)', maxent, score_maxent, usa_fondo=True),
    Modelo('Maxent (prospecciones)', maxent, score_maxent),
    Modelo('XGBoost',
           lambda y: XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01,
                                   subsample=0.8, colsample_bytree=0.8,
                                   scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
                                   random_state=42, n_jobs=-1, verbosity=0,
                                   eval_metric='auc'),
           proba),
    Modelo('Random Forest',
           lambda y: RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5,
                                            class_weight='balanced', random_state=42, n_jobs=-1),
           proba),
    Modelo('GLM',
           lambda y: make_pipeline(StandardScaler(),
                                   LogisticRegression(max_iter=1000, random_state=42)),
           proba),
]


In [ ]:
# ============================================================
# ESTRATEGIAS DE VALIDACIÓN
# ============================================================
def fuera_de_municipio(modelo):
    """Para cada municipio, entrena sin sus sitios y puntúa todos sus sitios candidatos."""
    pred_cand = np.full(len(candidatos), np.nan)
    por_mun = []
    for mun in candidatos['municipio'].unique():
        tr = grupos != mun
        en_mun = (candidatos['municipio'] == mun).values
        modelo.fit(X[tr], y[tr], grupos[tr])
        pred_cand[en_mun] = modelo.score(X_cand[en_mun])
        te = en_mun & observado
        y_te = candidatos.loc[te, 'D'].values
        if len(y_te) and y_te.min() < y_te.max():  # municipio con presencias y ausencias
            por_mun.append({'modelo': modelo.nombre, 'municipio': mun,
                            **metricas(y_te, pred_cand[te])})
    return pred_cand, pd.DataFrame(por_mun)

def reserva_municipios(modelo):
    te = np.isin(grupos, MUNICIPIOS_HOLDOUT)
    modelo.fit(X[~te], y[~te], grupos[~te])
    return metricas(y[te], modelo.score(X[te]))

def division_aleatoria(modelo):
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y)
    modelo.fit(X[tr], y[tr], grupos[tr])
    return metricas(y[te], modelo.score(X[te]))


In [ ]:
# ============================================================
# EVALUACIÓN
# ============================================================
METRICAS = ['auc', 'precision_at_k', 'recall_at_k', 'lift_at_k', 'brier']
filas, por_municipio = [], []
pred_oof = pd.DataFrame({'sitio': candidatos['sitio'], 'municipio': candidatos['municipio'],
                         'lon': candidatos['lon'], 'lat': candidatos['lat'], 'label': candidatos['D'],
                         'n_exhumaciones_hallazgo': candidatos['n_exhumaciones_hallazgo']})
pred_oof[nombres_covariables] = X_cand

for modelo in MODELOS:
    pred_cand, df_mun = fuera_de_municipio(modelo)
    pred_oof[modelo.nombre] = pred_cand
    pred = pred_cand[observado]
    por_municipio.append(df_mun)

    filas.append({'modelo': modelo.nombre, 'estrategia': 'Fuera de municipio (promedio)',
                  **df_mun[METRICAS].mean().to_dict(),
                  'prevalencia': df_mun['prevalencia'].mean(), 'n': len(df_mun)})
    # El agrupado se calcula como referencia, pero no se reporta: al dejar por fuera un municipio
    # grande con alta tasa de hallazgo baja la tasa de entrenamiento justo cuando se lo predice
    # (un modelo nulo obtiene un AUC agrupado de 0.32).
    filas.append({'modelo': modelo.nombre, 'estrategia': 'Fuera de municipio (agrupado)',
                  **metricas(y, pred)})
    filas.append({'modelo': modelo.nombre, 'estrategia': 'Reserva de municipios',
                  **reserva_municipios(modelo)})
    filas.append({'modelo': modelo.nombre, 'estrategia': 'División 80/20',
                  **division_aleatoria(modelo)})
    print(f"✓ {modelo.nombre}")

df_metricas = pd.DataFrame(filas)
# La salida de Maxent con fondo no es una probabilidad de hallazgo en la muestra intervenida
df_metricas.loc[df_metricas['modelo'] == 'Maxent (fondo)', 'brier'] = np.nan
df_por_municipio = pd.concat(por_municipio, ignore_index=True)

df_metricas.to_csv(Path(PATH_PROCESSED) / "metricas_modelos.csv", index=False)
df_por_municipio.to_csv(Path(PATH_PROCESSED) / "metricas_modelos_por_municipio.csv", index=False)
pred_oof.to_csv(Path(PATH_PROCESSED) / "predicciones_fuera_municipio.csv", index=False)

pd.set_option('display.width', 200)
print(df_metricas.round(3).to_string(index=False))


In [ ]:
# ============================================================
# CUADRO PARA LA TESIS
# ============================================================
estrategias = ['Fuera de municipio (promedio)', 'Reserva de municipios', 'División 80/20']
for metrica in METRICAS:
    tabla = (df_metricas[df_metricas['estrategia'].isin(estrategias)]
             .pivot(index='modelo', columns='estrategia', values=metrica)
             .loc[[m.nombre for m in MODELOS], estrategias])
    print(f"\n{metrica}")
    print(tabla.round(3).to_string())


In [ ]:
# ============================================================
# RECONCILIACIÓN CON LAS CIFRAS ANTERIORES
# ============================================================
# Las cifras anteriores de P@k fuera de municipio promediaban también los municipios
# que solo tienen presencias, donde P@k = 1 por construcción.
for modelo in MODELOS:
    p_k = []
    for mun in np.unique(grupos):
        idx = grupos == mun
        if y[idx].sum() == 0:
            continue
        p_k.append(metricas(y[idx], pred_oof.loc[observado, modelo.nombre].values[idx])['precision_at_k'])
    print(f"{modelo.nombre:<15} P@k promedio incluyendo municipios solo con presencias "
          f"(n={len(p_k)}): {np.mean(p_k):.3f}")


In [ ]:
# ============================================================
# ROBUSTEZ: DILIGENCIAS CON NDVI EN 0
# ============================================================
# Algunos sitios tienen NDVI faltante (imputado en 0) o exactamente 0, mientras que
# ningún punto de fondo lo tiene. Se verifica que esto no explique el desempeño de
# Maxent con fondo, repitiendo el AUC agrupado sin esas diligencias.
ndvi_cero = X[:, 0] == 0
print(f"Sitios con NDVI en 0: {ndvi_cero.sum()} (tasa de hallazgo {y[ndvi_cero].mean():.2f})")
for modelo in MODELOS:
    s = pred_oof.loc[observado, modelo.nombre].values
    print(f"{modelo.nombre:<24} AUC agrupado: {roc_auc_score(y, s):.3f}  |  "
          f"sin NDVI en 0: {roc_auc_score(y[~ndvi_cero], s[~ndvi_cero]):.3f}")
